# Phase 8A - UCI Raw URL Dataset Analysis

This notebook inspects the official PhiUSIIL CSV before URL-only feature engineering or model training. It does not alter Phase 1-7 artifacts or train a model.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'dataset' / 'raw' / 'PhiUSIIL_Phishing_URL_Dataset.csv'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)
df = pd.read_csv(DATA_PATH)
url_column = 'URL'
label_column = 'label'
assert url_column in df.columns and label_column in df.columns
assert set(df[label_column].unique()) == {0, 1}
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Label mapping verified: 0 = Phishing, 1 = Legitimate')
print('Sample labels (URL strings withheld):')
display(df[[label_column]].head())

Shape: (235795, 56)
Columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']
Label mapping verified: 0 = Phishing, 1 = Legitimate

## Quality, labels, domains, and leakage indicators

In [2]:
domain_column = 'Domain' if 'Domain' in df.columns else None
timestamp_candidates = [c for c in df.columns if any(t in c.lower() for t in ('time','date','timestamp'))]
summary = {
    'rows': int(len(df)),
    'columns': int(df.shape[1]),
    'column_names': list(df.columns),
    'dtypes': {c: str(t) for c, t in df.dtypes.items()},
    'missing_values_total': int(df.isna().sum().sum()),
    'missing_values_by_column': {c: int(v) for c, v in df.isna().sum().items() if v},
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_urls': int(df[url_column].duplicated().sum()),
    'unique_urls': int(df[url_column].nunique()),
    'unique_domains': int(df[domain_column].nunique()) if domain_column else None,
    'label_mapping': {'0': 'Phishing', '1': 'Legitimate'},
    'class_distribution': {str(k): int(v) for k, v in df[label_column].value_counts().sort_index().items()},
    'domain_column': domain_column,
    'timestamp_candidates': timestamp_candidates,
    'potential_leakage_fields': [c for c in df.columns if c not in {url_column, label_column} and c in {'FILENAME','Domain','TLD','TLDLegitimateProb','URLSimilarityIndex','URLCharProb','Title','DomainTitleMatchScore','URLTitleMatchScore'}],
}
print('Data types:')
display(df.dtypes.rename('dtype').to_frame())
print('Class distribution:')
display(df[label_column].value_counts().sort_index().rename('count').to_frame())
print('Unique domains:', summary['unique_domains'])
print('Timestamp candidates:', timestamp_candidates)
(RESULTS / 'url_dataset_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved:', RESULTS / 'url_dataset_summary.json')

Data types:


,dtype
FILENAME,str
URL,str
URLLength,int64
Domain,str
DomainLength,int64
IsDomainIP,int64
TLD,str
URLSimilarityIndex,float64
CharContinuationRate,float64
TLDLegitimateProb,float64


Class distribution:


,count
label,
0,100945
1,134850


Unique domains: 220086
Timestamp candidates: []
Saved: ./results\url_dataset_summary.json


## URL-only feature policy

The URL model may use deterministic measurements from the raw URL string and parsed hostname/path/query. It excludes HTML, title, robots, redirects, DNS/WHOIS, reputation, search, traffic, and page-reference fields. The `Domain` column is used only as a grouping identifier for evaluation, not as a model feature.